In [1]:
import polars as pl
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import lightgbm as lgb

In [2]:
df = pl.scan_csv('./data/train_v2.csv',
                schema_overrides={
                    "fullVisitorId": pl.Utf8 
    })

In [3]:
df.head()

In [4]:
keep_cols = [
    "channelGrouping",
    "device",
    "geoNetwork",
    "trafficSource",
    "visitNumber",
    "visitStartTime",
    "date",
    "totals"
]

In [5]:
df = df.select(keep_cols)

In [6]:
# df = df.collect(engine="streaming")
# df.select("device").head(1).to_dicts()

In [7]:
# df = df.with_columns([
#     pl.col("device").str.parse_json().alias("device"),
#     pl.col("geoNetwork").str.parse_json().alias("geoNetwork"),
#     pl.col("totals").str.parse_json().alias("totals"),
# ])

In [8]:
# df = df.with_columns([
#     pl.col("device").struct.field("deviceCategory").alias("deviceCategory"),
#     pl.col("device").struct.field("isMobile").alias("isMobile"),

#     pl.col("geoNetwork").struct.field("country").alias("country"),
#     pl.col("geoNetwork").struct.field("subContinent").alias("subContinent"),

#     pl.col("totals").struct.field("pageviews").alias("pageviews"),
#     pl.col("totals").struct.field("transactionRevenue").alias("transactionRevenue"),
# ])

In [9]:
def safe_json(x):
    try:
        return json.loads(x)
    except:
        return {}

def column_split(df):

    device = pd.json_normalize(df["device"])
    device = device[["isMobile", "deviceCategory"]]

    geo = pd.json_normalize(df["geoNetwork"])
    geo = geo[["subContinent", "country"]]

    totals = pd.json_normalize(df["totals"])

    totals = totals.rename(columns={"hits": "hit"})

    keep_totals = [
        "visits",
        "hit",
        "pageviews",
        "transactions",
        "transactionRevenue"
    ]

    totals = totals[keep_totals].apply(pd.to_numeric, errors="coerce")

    df_new = pd.concat([
        df.drop(columns=["device", "geoNetwork", "totals", "trafficSource"]),
        device, geo, totals
    ], axis=1)

    return df_new

In [10]:
all_chunks = []
count = 0
for chunk in df.collect(engine="streaming").iter_slices(1000000):

    df_chunk = chunk.to_pandas()
    df_chunk["device"] = df_chunk["device"].apply(safe_json)
    df_chunk["geoNetwork"] = df_chunk["geoNetwork"].apply(safe_json)
    df_chunk["totals"] = df_chunk["totals"].apply(safe_json)

    df_chunk = column_split(df_chunk)
    count += len(df_chunk)

    all_chunks.append(df_chunk)

In [11]:
new_df = pd.concat(all_chunks, axis=0)

In [12]:
new_df.head()

,channelGrouping,visitNumber,visitStartTime,date,isMobile,deviceCategory,subContinent,country,visits,hit,pageviews,transactions,transactionRevenue
0,Organic Search,1,1508198450,20171016,False,desktop,Western Europe,Germany,1,1,1.0,NaN,NaN
1,Referral,6,1508176307,20171016,False,desktop,Northern America,United States,1,2,2.0,NaN,NaN
2,Direct,1,1508201613,20171016,True,mobile,Northern America,United States,1,2,2.0,NaN,NaN
3,Organic Search,1,1508169851,20171016,False,desktop,Western Asia,Turkey,1,2,2.0,NaN,NaN
4,Organic Search,1,1508190552,20171016,False,desktop,Central America,Mexico,1,2,2.0,NaN,NaN


In [14]:
new_df.shape

(1708337, 13)

In [15]:
count

1708337